# Single-DB Text-to-SQL Training Story

This notebook is an interview-defense artifact for the storefront single-database text-to-SQL lane. The story is not "I trained a general SQL model." The story is: I trained and evaluated a schema-specialized model for one production-style database, then used controlled experiments to decide which data, adapter, quantization, and serving choices were defensible.

Use this notebook to rehearse a deep dive with a senior applied scientist interviewer. The important posture is evidence over vibes: fixed evaluation gates, one changed variable per experiment, failure analysis, explicit rejection of regressions, and no official benchmark claims.

## 90-Second Version

I worked on a bounded text-to-SQL problem: one known database, stable business semantics, and repeated analyst-style questions over that same schema. So I did not optimize for cross-database generalization. I optimized for same-database transfer: can the model answer new business questions over a fixed schema without copying held-out answers?

I built a manifest-driven LoRA SFT pipeline around `Qwen/Qwen3.5-0.8B-Base`, froze dev and held-out eval sets, and evaluated by executing generated SQL against a fixed SQLite database snapshot. The baseline model got 1/12 on dev and 1/12 on held-out eval. A first 40-row LoRA run improved dev to 8/12 but only eval to 3/12, which told me the model was learning local patterns but not robust same-DB composition.

The main failures were not generic SQL syntax. They were schema ownership and business semantics: putting `quantity` on `orders` instead of `order_items`, using the wrong alias for `returns.return_id`, mishandling anti-joins and HAVING, and inventing filters like `issue_type = 'support'`. I expanded the train set from 40 to 97 to 130 rows, but not randomly. I added execution-checked examples around observed failure families while keeping dev/eval frozen.

Exp048 proved targeted train_v3 data worked: same LoRA recipe as the prior best, 10/12 dev and 10/12 held-out eval. Then I stress-tested that result with a fresh same-DB challenge set and broke the next work into ablations. Exp051-Exp055 isolated support filters, date boundaries, return ratios, HAVING, and anti-joins. The best isolated ablations reached 17/24 challenge, but the real jump came from composing them: Exp056 train_v4 LoRA reached 11/12 dev_v2, 12/12 eval_v1, and 22/24 challenge_v1.

I also tested QLoRA under the same train_v4 data. Exp057 trained faster and loaded fewer total parameters, reached 12/12 dev_v2 and 22/24 challenge_v1, but regressed stable eval_v1 to 10/12. So my promoted quality checkpoint is Exp056 LoRA. QLoRA is a working memory/runtime path, not the preferred model for this one-DB story.

## What The Problem Was And Was Not

**What it was:** a production-style single-database specialization problem. The schema is allowed to be learned. The join paths, metric definitions, common filters, and business vocabulary are allowed to become part of the adapter behavior.

**What it was not:** a generalized BIRD/Spider/LiveSQLBench model. There is no DB-level holdout in the storefront lane because the point is one database. The correct gate is unseen questions and unseen compositions over the same database.

The leakage boundary is therefore different:

- Schema memorization is acceptable.
- Business semantic learning is acceptable.
- Held-out question or held-out SQL memorization is not acceptable.
- New train rows must teach mechanisms, not copy eval answers.
- Promotion requires frozen dev/eval execution results, not training loss.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

ROOT = Path.cwd()
EXPERIMENTS = ROOT / "experiments" / "sql"
ARTIFACTS = ROOT / "artifacts" / "sql"
RESULTS = ROOT / "results" / "sql"
DATASETS = ROOT / "datasets" / "sql"

STORE_EXPS = [
    "qwen35_0_8b__exp039_storefront_single_db_lab",
    "qwen35_0_8b__exp040_storefront_from_exp031_sql_adapter",
    "qwen35_0_8b__exp041_storefront_v2_from_exp031_sql_adapter",
    "qwen35_0_8b__exp042_storefront_v2_continue_exp040",
    "qwen35_0_8b__exp043_storefront_v2_lora_r8_a16_d005",
    "qwen35_0_8b__exp044_storefront_v2_lora_r16_a32_d005",
    "qwen35_0_8b__exp046_storefront_v2_lora_r16_a32_d010",
    "qwen35_0_8b__exp048_storefront_v3_lora_r16_a32_d010",
    "qwen35_0_8b__exp049_storefront_v3_qlora_r16_a32_d010",
]

def load_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def jsonl_rows(path: Path) -> list[dict]:
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]

def short_exp(exp: str) -> str:
    marker = "__exp"
    return "Exp" + exp.split(marker, 1)[1][:3]


## Dataset And Gate Design

The dataset design is the first thing to defend. Since this is one database, `train`, `dev`, and `eval` all share the database. That is intentional. The guard is frozen held-out questions and SQL, not DB-disjoint schemas.

- `train_v1`: 40 broad local examples.
- `train_v2`: 97 examples, adding 57 targeted non-heldout examples for column ownership, join paths, anti-joins, HAVING, ratios, shipments, returns, and support tickets.
- `train_v3`: 130 examples, adding 33 execution-checked examples around the remaining composition failures.
- `train_exp051_support_v1`: 144 examples, equal to train_v3 plus 14 support-ticket no-issue-filter rows.
- `train_exp052_date_v1`: 144 examples, equal to train_v3 plus 14 date-boundary rows.
- `train_exp053_return_ratio_v1`: 144 examples, equal to train_v3 plus 14 return-ratio and alias-ownership rows.
- `train_exp054_having_v1`: 144 examples, equal to train_v3 plus 14 grouped-count and HAVING rows.
- `train_exp055_antijoin_v1`: 144 examples, equal to train_v3 plus 14 anti-join and missing-relationship rows.
- `train_v4`: 200 examples, combining train_v3 plus all five targeted supplements.
- `dev_v1`: 12 frozen quick-gate cases.
- `dev_v2`: 12 targeted same-DB cases for the new failure families.
- `eval_v1`: 12 frozen held-out cases.
- `challenge_v1`: 24 fresh same-DB stress cases with no exact train_v4 question or gold-SQL overlap.

The key interview phrase: **same DB is allowed; copied held-out answers are not.**


In [ ]:
dataset_paths = [
    DATASETS / "train" / "storefront_sales_lab_train_v1.jsonl",
    DATASETS / "train" / "storefront_sales_lab_train_v2.jsonl",
    DATASETS / "train" / "storefront_sales_lab_train_v3.jsonl",
    DATASETS / "train" / "storefront_sales_lab_train_exp051_support_v1.jsonl",
    DATASETS / "train" / "storefront_sales_lab_train_exp052_date_v1.jsonl",
    DATASETS / "train" / "storefront_sales_lab_train_exp053_return_ratio_v1.jsonl",
    DATASETS / "train" / "storefront_sales_lab_train_exp054_having_v1.jsonl",
    DATASETS / "train" / "storefront_sales_lab_train_exp055_antijoin_v1.jsonl",
    DATASETS / "train" / "storefront_sales_lab_train_v4.jsonl",
    DATASETS / "eval" / "storefront_sales_lab_dev_v1.jsonl",
    DATASETS / "eval" / "storefront_sales_lab_dev_v2.jsonl",
    DATASETS / "eval" / "storefront_sales_lab_eval_v1.jsonl",
    DATASETS / "eval" / "storefront_sales_lab_challenge_v1.jsonl",
]

for path in dataset_paths:
    rows = jsonl_rows(path)
    tag_counts = {}
    for row in rows:
        for tag in row.get("tags", []):
            tag_counts[tag] = tag_counts.get(tag, 0) + 1
    top_tags = sorted(tag_counts.items(), key=lambda item: item[1], reverse=True)[:10]
    print(path.name, "rows=", len(rows), "top_tags=", top_tags)


## Why This Base Model

The base model was `Qwen/Qwen3.5-0.8B-Base` because the product constraint was small-model specialization, not highest possible benchmark score. For one known database, a small model can be viable if the adapter learns schema-specific structure and the serving cost stays low.

Defensible reasons:

- It is cheap enough to iterate locally.
- It keeps the experiment loop fast enough for failure-driven data work.
- The task is bounded: one schema, one dialect, fixed prompt shape.
- The company story is about controlled specialization and deployment constraints, not chasing a leaderboard.

Rejected framing: "I used a small model because it was best." The stronger framing is: "I used a small model because the task boundary made specialization plausible, and I measured whether it was enough."

## Why LoRA First

The first promoted path is regular LoRA SFT. This is the cleanest controlled choice for a small model:

- Freeze the base model.
- Train a small PEFT adapter.
- Keep the artifact portable.
- Iterate without full model checkpoint management.
- Compare adapter recipes by changing one knob at a time.

The adapter target modules were the standard attention and MLP projection modules: `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, and `down_proj`.

The loss target was assistant SQL only. The model is not trained to explain. It is trained to emit the final SQL statement.

In [ ]:
def manifest_for(exp: str) -> dict:
    path = EXPERIMENTS / f"{exp}.json"
    return load_json(path)

for exp in STORE_EXPS:
    manifest = manifest_for(exp)
    trainer = manifest.get("trainer", {})
    lora = manifest.get("lora", {})
    quant = manifest.get("quantization", {"mode": "none"})
    print(
        short_exp(exp),
        manifest["training_method"]["method"],
        "train=", ",".join(Path(p).name for p in manifest["train_inputs"]["train_datasets"]),
        "epochs=", trainer.get("num_train_epochs"),
        "lr=", trainer.get("learning_rate"),
        "lora=", f"r{lora.get('r')}/a{lora.get('lora_alpha')}/d{lora.get('lora_dropout')}/b{lora.get('bias', 'default')}",
        "quant=", quant.get("mode"),
    )


## Experiment Narrative

This is the core interview sequence.

### Exp039: prove the local specialization path

Exp039 trained from the base model on 40 storefront rows with LoRA r8/alpha16/dropout0.05 for 3 epochs. It moved dev from 1/12 to 8/12, but held-out eval only to 3/12. That was useful but not promotable. It proved the adapter could learn local SQL idioms, but also showed overfit to dev-like shapes.

### Exp040: test warm-start from a prior SQL adapter

Exp040 continued from the Exp031 SQL/profile-metadata adapter. Same data and hyperparameters as Exp039. It improved dev to 11/12 and eval to 4/12. The prior SQL adapter helped, but it did not solve held-out composition. Decision: useful baseline, not enough.

### Exp041/042: expand data and try schedule corrections

Train_v2 expanded from 40 to 97 rows, keeping dev/eval frozen. Exp041 trained one epoch from Exp031 and underfit: 6/12 dev, 3/12 eval. Exp042 continued from the stronger Exp040 with lower LR, preserving more dev behavior but still only 3/12 eval. Lesson: clean broader data is necessary but not sufficient; schedule and representation matter.

### Exp043/044/046: isolate LoRA shape

Using train_v2 from the base model, the lane swept LoRA capacity and dropout. r8/alpha16 reached 4/12 eval, r16/alpha32 reached 5/12, and r16/alpha32/dropout0.10 reached 6/12. Capacity and regularization mattered, but the remaining failures were still composition and ownership errors.

### Exp048: targeted train_v3 data wins

Exp048 kept the best Exp046 LoRA recipe fixed and changed only train_v2 to train_v3. Train_v3 added 33 execution-checked examples for grouped ranking, return ratios, anti-join lists, global HAVING, shipment-return joins, and explicit alias ownership. Held-out eval jumped from 6/12 to 10/12. This became the first promoted local HF adapter.

### Exp049: QLoRA ablation works, but quality regresses

Exp049 repeated Exp048 with QLoRA only: bitsandbytes 4-bit NF4, double quantization, bfloat16 compute intent, device_map auto, and PEFT k-bit preparation. It trained and served, but scored 9/12 dev and 9/12 eval, below Exp048.

### Exp050-055: isolate the remaining one-DB failure families

Exp050 evaluated Exp048 on a fresh 24-case same-DB challenge and got 15/24. Then Exp051-Exp055 each added exactly one 14-row supplement over train_v3: support no-issue filters, date boundaries, return ratios, HAVING, and anti-joins. The best isolated result was useful but limited: Exp053, Exp054, and Exp055 each reached 17/24 challenge in different ways, with different failure families.

### Exp056: train_v4 composition is the quality jump

Exp056 combined all five supplements into train_v4 with 200 rows, keeping the Exp048 LoRA recipe fixed. It reached 11/12 dev_v2, 12/12 eval_v1, and 22/24 challenge_v1. The challenge failures were only schema-format failures, not row-count or row-value mismatches. This is the promoted LoRA checkpoint.

### Exp057: train_v4 QLoRA is an infra tradeoff, not the quality winner

Exp057 repeated Exp056 with QLoRA only. It trained faster, loaded fewer total parameters, reached 12/12 dev_v2 and matched 22/24 challenge_v1, but regressed eval_v1 to 10/12. Decision: keep QLoRA as a credible memory/runtime path; do not promote it over Exp056.

## Exp050-057: Break The Next Work Into Experiments

After Exp048 and Exp049, the right move was not one vague larger fine-tune. It was a ladder where each experiment answered one question.

| Experiment | Change | Key result | Decision |
|---|---|---|---|
| Exp050 | Evaluate Exp048 on `challenge_v1` only | 15/24 challenge | Establish fresh same-DB stress baseline. |
| Exp051 | Add support no-issue-filter examples | 6/12 dev_v2, 9/12 eval, 13/24 challenge | Reject as replacement; support cases helped narrowly but broader behavior regressed. |
| Exp052 | Add date-boundary examples | 6/12 dev_v2, 10/12 eval, 14/24 challenge | Reject as replacement; preserves eval but does not beat challenge baseline. |
| Exp053 | Add return-ratio examples | 8/12 dev_v2, 10/12 eval, 17/24 challenge | Best isolated ablation; useful evidence for train_v4. |
| Exp054 | Add grouped HAVING examples | 9/12 dev_v2, 9/12 eval, 17/24 challenge | Reject as replacement; targeted gain but eval regression. |
| Exp055 | Add anti-join examples | 9/12 dev_v2, 10/12 eval, 17/24 challenge | Useful but not better than Exp053. |
| Exp056 | Combine Exp051-Exp055 into train_v4 | 11/12 dev_v2, 12/12 eval, 22/24 challenge | Promote as best LoRA checkpoint. |
| Exp057 | Repeat Exp056 with QLoRA | 12/12 dev_v2, 10/12 eval, 22/24 challenge | Strong infra tradeoff; reject as preferred checkpoint. |

The defendable story is that every row family is an intervention. The isolated ablations diagnose what helps, but promotion waits for the combined run to prove the families compose across targeted dev, stable eval, and fresh challenge gates.

In [ ]:
def result_scores(exp: str) -> list[tuple[str, str, str]]:
    root = RESULTS / exp
    rows = []
    if not root.exists():
        return rows
    for path in sorted(root.glob("*.json")):
        if path.name.endswith(".analysis.json"):
            continue
        data = load_json(path)
        if "case_count" not in data:
            continue
        dataset = Path(str(data.get("eval_dataset"))).name
        score = f"{data.get('passed_count')}/{data.get('case_count')}"
        rows.append((path.name, dataset, score))
    return rows

for exp in STORE_EXPS:
    scores = result_scores(exp)
    if not scores:
        continue
    print("\n", short_exp(exp), exp)
    for name, dataset, score in scores:
        print(" ", score, dataset, name)


In [ ]:
for exp in STORE_EXPS:
    path = ARTIFACTS / exp / "train_summary.json"
    if not path.exists():
        continue
    summary = load_json(path)
    metrics = summary.get("training_metrics") or {}
    print(
        short_exp(exp),
        "rows=", summary.get("train_row_count"),
        "runtime=", metrics.get("train_runtime"),
        "loss=", metrics.get("train_loss"),
        "trainable=", summary.get("trainable_parameters"),
        "total=", summary.get("total_parameters"),
    )


## Failure Analysis Is The Story

A senior interviewer will not be satisfied with the score curve. They will ask: what failed, why did it fail, and what did you do next?

The answer is that the failures became the curriculum.

- Column ownership: revenue formulas sometimes put `quantity` or `unit_price` on the wrong table.
- Alias ownership: the model joined the right table but counted from the wrong alias.
- Return ratios: the denominator and numerator needed different tables and a left join.
- Anti-joins: the model produced invalid double-WHERE patterns before targeted anti-join examples.
- HAVING/count logic: the model counted all customers instead of grouped customers with more than one completed order.
- Semantic overfiltering: unresolved support tickets should filter `resolved = 0`, not invent `issue_type = 'support'`.

The best phrasing: **I did not add data to make the dataset bigger. I added data to attack named failure mechanisms.**

In [ ]:
case_by_id = {}
for path in [DATASETS / "eval" / "storefront_sales_lab_dev_v1.jsonl", DATASETS / "eval" / "storefront_sales_lab_eval_v1.jsonl"]:
    for row in jsonl_rows(path):
        case_by_id[row["case_id"]] = row

def print_failures(result_path: Path) -> None:
    data = load_json(result_path)
    print(result_path)
    print("score", f"{data.get('passed_count')}/{data.get('case_count')}")
    for record in data.get("records", []):
        if record.get("passed"):
            continue
        case = case_by_id.get(record.get("case_id"), {})
        print("\nCASE", record.get("case_id"), "tags=", case.get("tags"))
        print("Q:", case.get("question"))
        print("ERR:", record.get("prediction_error"))
        print("PRED:", record.get("predicted_sql"))
        print("GOLD:", case.get("gold_sql"))
        print("pred_rows:", record.get("predicted_rows"))
        print("gold_rows:", record.get("gold_rows"))

print_failures(RESULTS / "qwen35_0_8b__exp048_storefront_v3_lora_r16_a32_d010" / "adapter__storefront_sales_lab_eval_v1__adapter_eval.json")


In [ ]:
print_failures(RESULTS / "qwen35_0_8b__exp049_storefront_v3_qlora_r16_a32_d010" / "adapter__storefront_sales_lab_eval_v1__eval.json")


## Why QLoRA, And Why It Was Rejected

QLoRA was not introduced as a magic quality improvement. It was introduced as a memory-path experiment.

The controlled question was: if I keep the same data, same prompt, same LoRA rank, same LR, same epochs, and same eval gates, can I train the adapter with a 4-bit-loaded base model and preserve quality?

Answer across the two QLoRA runs:

- Exp049 QLoRA on train_v3 worked operationally but regressed quality: 9/12 dev and 9/12 eval versus Exp048 LoRA at 10/12 and 10/12.
- Exp057 QLoRA on stronger train_v4 was much more competitive: 12/12 dev_v2 and 22/24 challenge_v1, matching or beating Exp056 on those gates.
- Exp057 still regressed the stable eval gate: 10/12 eval_v1 versus Exp056 LoRA at 12/12.
- Exp057 was operationally attractive: runtime `2013.6028s` versus Exp056 `2229.787s`, and loaded total parameters `561,809,472` versus `859,375,680`.

The defensible interview claim is narrow:

> QLoRA is useful when memory or runtime is the binding constraint, especially to unlock a larger base model or longer context. In this 0.8B one-DB lab, QLoRA became a credible infra alternative after train_v4, but regular LoRA remained the promoted quality path because it preserved every frozen quality gate.

This is a good scientific result because a plausible technique was tried twice under fixed gates: first rejected on train_v3 quality, then retained as an efficiency tradeoff on train_v4 but still not promoted.

## Serving And Infra Story

The serving story matters because a model is not done when local evaluation passes. Exp048 local HF scored 10/12, but the vLLM endpoint scored 9/12. That makes endpoint parity part of the model contract.

The local vLLM runtime needed compatibility flags on WSL RTX 2080 Ti:

- `gpu_memory_utilization=0.75`
- eager mode
- FlashInfer sampler disabled
- FlashInfer autotune disabled
- `TRITON_ATTN`
- OpenAI-compatible completions endpoint

The transport worked. The quality gate did not fully pass. Therefore the endpoint was not promoted.

This is a strong infra answer: **serving success, latency, and model quality are separate gates.**

In [ ]:
def stress_rows(exp: str) -> list[tuple]:
    rows = []
    for path in sorted((ARTIFACTS / exp).glob("vllm_stress_*_r*.json")):
        data = load_json(path)
        records = data.get("records", [])
        chars = [r.get("generated_char_count") for r in records if r.get("generated_char_count") is not None]
        avg_chars = round(sum(chars) / len(chars), 1) if chars else None
        rows.append((
            path.name,
            data.get("concurrency"),
            data.get("request_count"),
            f"{data.get('success_count')}/{data.get('request_count')}",
            round(data.get("requests_per_second"), 4),
            round(data.get("p50_latency_seconds"), 2),
            round(data.get("p95_latency_seconds"), 2),
            avg_chars,
        ))
    return rows

for exp in [
    "qwen35_0_8b__exp048_storefront_v3_lora_r16_a32_d010",
    "qwen35_0_8b__exp049_storefront_v3_qlora_r16_a32_d010",
]:
    print("\n", exp)
    for row in stress_rows(exp):
        print(row)


## Alternatives Tried And Rejected

A deep-dive interview needs alternatives. These are the ones to name.

### Prompting only

Rejected as insufficient for repeated schema-specific semantics. Prompting can help simple grounding, but the failures were stable ownership and composition errors. The model needed training signal for recurring patterns.

### Warm-start from a prior SQL adapter

Tried in Exp040. It improved dev strongly and eval slightly, so it was useful but not sufficient. It became evidence that prior SQL behavior helps, but schema-specific train data still matters.

### More data without enough schedule/capacity control

Tried around train_v2. Clean broader data did not automatically improve eval. One epoch underfit; continuation preserved dev but did not improve held-out eval. The lesson was to isolate data, schedule, and adapter capacity.

### LoRA shape sweep

Useful. r16/alpha32/dropout0.10 was better than the r8 baseline on held-out eval. But it was still not enough until targeted train_v3 data was added.

### Isolated row-family supplements

Useful for diagnosis, not directly promotable. Exp053/054/055 each reached 17/24 challenge but moved failures around. The result justified train_v4 rather than a single-family checkpoint.

### QLoRA

Useful as infra, rejected as preferred quality checkpoint. Exp057 trained faster and matched challenge, but regressed stable eval_v1 from 12/12 to 10/12.

### vLLM endpoint promotion

Rejected for now. Endpoint quality was 9/12 versus local HF 10/12 for Exp048. Transport success is not promotion.

## The Full Interview Answer

If asked to go deep, answer in this order.

### 1. Frame the task boundary

> This was a single production-database text-to-SQL model, not a general SQL model. I allowed the model to learn schema and business semantics, but I protected held-out question and SQL leakage. The metric was new questions over the same fixed schema.

### 2. Explain the data split

> I kept frozen dev and held-out eval files. Train/dev/eval shared the DB by design, but had distinct task IDs, questions, and SQL. I audited exact leakage and evaluated by execution result equivalence. After Exp049, I added `dev_v2` and `challenge_v1` as fresh same-DB gates and broke train_v4 into five controlled supplements before combining them.

### 3. Explain model and method choice

> I chose Qwen3.5 0.8B because the production task was bounded and local iteration mattered. I used LoRA first because it gave a small, reproducible adapter without full-model fine-tuning. The trainer was TRL SFTTrainer, with assistant-SQL-only loss, canonical chat prompt, and manifest-pinned settings.

### 4. Walk the score curve

> Base was 1/12. The first 40-row LoRA reached 8/12 dev but only 3/12 eval. Warm-start improved dev to 11/12 and eval to 4/12. LoRA capacity/dropout got eval to 6/12. Targeted train_v3 data with the same best LoRA recipe reached 10/12 dev and 10/12 eval in Exp048. Then train_v4 composition reached 11/12 dev_v2, 12/12 eval_v1, and 22/24 challenge_v1 in Exp056.

### 5. Name the failures

> The failures were column ownership, alias ownership, return-ratio joins, anti-joins, HAVING, date boundaries, and semantic overfiltering. For example, the model learned the revenue formula but sometimes put `quantity` on the wrong alias. That is why I added targeted examples for those mechanisms.

### 6. Explain the Exp050-057 design

> I did not expand the single-DB data blindly. I split the next work into one intervention per failure family: support filters, date boundaries, return ratios, HAVING, and anti-joins. Each got a 144-row ablation, then a 200-row combined train_v4 run. The isolated ablations topped out at 17/24 challenge, while the combined run reached 22/24 and eliminated semantic challenge mismatches.

### 7. Explain QLoRA honestly

> QLoRA was an infrastructure hypothesis, not a promised quality win. On train_v4 it ran faster and used a 4-bit loaded base model, and it got 12/12 dev_v2 plus 22/24 challenge. But it regressed stable eval_v1 from 12/12 to 10/12, so I kept QLoRA as a memory/runtime option and promoted regular LoRA.

### 8. State the final decision

> The promoted checkpoint is Exp056 LoRA on train_v4. The practical lesson is that the dataset design drove the quality jump, and the framework choice was evaluated as a controlled tradeoff rather than assumed from popularity.

## Questions An Interviewer Will Ask

### Why is one-DB training scientifically valid?

Because the product target is one DB. The scientific question is not schema generalization. It is composition generalization over a fixed schema. The correct holdout is frozen same-DB business questions, not unseen databases.

### How did you prevent leakage?

I kept dev/eval frozen, used distinct task IDs/questions/SQL, audited exact leakage, and added train rows by failure mechanism rather than copying held-out answers. For train_v4, I also check no exact question or SQL overlap against dev_v1, dev_v2, eval_v1, and challenge_v1.

### How else would you diversify train/dev/eval for one DB?

Diversify by semantic mechanism, not by pretending to have new schemas: boundary dates, grouped thresholds, null and missing relationships, ratio denominators, alias ownership, synonym phrasing, multi-hop join paths, and value-level perturbations. Keep each family isolated before combining it, and keep fresh same-DB gates for promotion.

### Why did train loss not decide promotion?

Because SQL behavior can regress even with good loss. Promotion used execution result equivalence on frozen gates.

### Why did Exp048 beat Exp046?

Same LoRA recipe, better targeted data. Exp048 changed the data from train_v2 to train_v3, adding examples around grouped ranking, return ratios, anti-joins, HAVING, shipment-return joins, and alias ownership.

### Why did QLoRA regress if train loss was similar?

Similar loss does not guarantee identical decoded SQL behavior. QLoRA changes the training load path and quantized base representation during adapter learning. Exp057 was usable and strong on dev_v2/challenge, but the frozen eval exposed two schema-format regressions versus Exp056.

### Why not promote vLLM if all load tests passed?

Because load success is not model correctness. vLLM quality was 9/12 where local HF was 10/12, so endpoint parity failed.

### What would make this production-ready?

Frozen quality gates, endpoint parity, SQL safety controls, read-only default execution, schema drift detection, query logging, confidence/fallback policy, and human review for risky queries.


## Latest Contrast Ablation: Exp058-062

The next question was not whether to make the model more general. It was how to diversify a one-DB model without copying held-out answers. I split that into hard-negative rows and richer eval slicing.

- Exp058 added per-tag slicing so I could stop treating one aggregate pass rate as the whole story.
- Exp063-style `challenge_v2` stressed alias ownership, boundary semantics, and anti-join / left-join predicate ownership. The promoted Exp056 checkpoint scored 8/15 on that fresh stress set.
- Exp059 added 12 alias-contrast rows: 12/12 dev_v2, 12/12 eval_v1, 22/24 challenge_v1, 9/15 challenge_v2. It was useful but narrow.
- Exp060 added 12 boundary rows: 12/12 dev_v2, 11/12 eval_v1, 23/24 challenge_v1, 9/15 challenge_v2. It improved targeted behavior but failed the protected eval gate.
- Exp061 added 12 anti-join rows: 12/12 dev_v2, 11/12 eval_v1, 23/24 challenge_v1, 8/15 challenge_v2. It moved the anti-join slice but transferred errors elsewhere.
- Exp062 combined all 36 rows into `train_v5` with 236 rows: 12/12 dev_v2, 11/12 eval_v1, 23/24 challenge_v1, 12/15 challenge_v2. It improved the fresh hard set but still regressed eval_v1.

Decision: Exp062 is not promoted. It is the ablation that proves the rule: slice gains are not promotion unless the stable holdout stays clean. The promoted checkpoint remains Exp056 for the one-DB story.


## Claims To Avoid

- Do not say this is a generalized text-to-SQL model.
- Do not report local same-DB results as official benchmark scores.
- Do not say QLoRA is the promoted checkpoint. Exp057 matched challenge and improved dev_v2, but regressed stable eval_v1.
- Do not say vLLM is promoted. It is mechanically wired but failed quality parity.
- Do not say Exp062 is promoted. It improved `challenge_v2` to 12/15 but regressed `eval_v1` to 11/12.
- Do not say more data solved it. Isolated failure-family data diagnosed the gaps; combined train_v4 proved the families composed.
- Do not claim 160 concurrent users without the latency context. Exp048 admitted c160, but p95 was about 63s in the earlier stress test.

The high-signal claim is: **I used controlled training science to turn a one-DB SQL problem into a measurable specialization system, promoted Exp056 only after frozen gates improved, and kept QLoRA as an infra tradeoff rather than overstating it.**